<div style="
    font-family: 'Arial', 'Calibri', sans-serif; 
    font-size: 11pt; 
    line-height: 1.5; 
    padding: 2.5cm;
    text-align: justify;
">
    
# Part 1 · Diagnosis and improvement plan

## 0. Context

Deep Learning-based **financial transaction fraud detection** system that has degraded its performance in production.

## Task 1 · Model and data diagnosis

### 1. Current system

- The architecture is a feedforward autoencoder that compresses the data into a lower-dimensional space trained with legitimate data, expecting frauds to cause problems when decoded. MSE is used as the error.
- Preprocessing pipeline: median/mode imputation, basic encoding and scaling, fixed threshold on reconstruction error to flag anomalies.

This design worked in its initial history, but did not incorporate robust mechanisms for adaptation to distribution change or advanced monitoring.

---

### 2. Current performance and symptoms

Among the observed symptoms are a declining fraud recall, that is, frauds escape without being flagged; more false positives in certain geographic circumstances and digital channels; and drift due to changes in consumption habits and new fraud strategies that camouflage themselves with real behavior, confusing the model. This has a direct impact: economic losses from undetected frauds, overload of the review team, and bad customer experience due to false alarms.

---

### 3. Data audit

Key points:

1. **Imputation and quality**: univariate imputation (median/mode) is used, which ignores relationships between variables and can introduce artificial profiles; there are strong outliers in amounts and counters without specific treatment.

2. **Representativeness and imbalance**: high fraud/non-fraud disproportion (typical case); changes in channel composition and commerce types; the training dataset no longer represents the current reality well.

3. **Segment biases**: higher false positive rate in certain groups (e.g. specific regions or channels), indicating that the model generalizes worse to new segments.

Conclusion: the model was trained on data that is no longer representative and with too simple preprocessing for a fast-evolving fraud environment.

---

### 4. Diagnostic metrics

Given the imbalance, overall accuracy is neither reliable nor a priority, or at least not as much. The following are selected:

- **Fraud recall**, we want to avoid undetected fraud-
- **Fraud precision**, we do not want to affect legitimate users.
- **Fraud F1**, balance between the previous two.
- **ROC-AUC and, above all, PR-AUC** (better reflection of performance in the minority class with PR-AUC).

These metrics serve as a baseline to evaluate any subsequent improvement.

---

## Task 2 · Improvement plan and technical redesign

---

### 1. Data improvement: k-NN imputation

We propose performing a k-NN imputation on numerical and encoded variables over the separated training and test data, because k-NN imputation better respects the relationships between variables (amount, frequency, channel, etc.), filling missing values with "similar" examples, which helps the autoencoder (and the future VAE) to learn a more faithful normality space, key in anomaly detection.

---

### 2. Architecture improvement: Variational Autoencoder (VAE)

We propose replacing the deterministic autoencoder with a **Variational Autoencoder** for the improved model, because the classic autoencoder learns a latent embedding without explicit structure; the VAE forces the latent space to follow a more regular distribution (e.g. Gaussian), which usually produces more continuous and robust representations, especially against subtle changes, which would be more suitable to prevent drift from causing unexpected problems. This can improve the separation between normal and anomalous transactions when reconstruction error is used as a score. For example, we propose the following design:
- Encoder:
  - Two dense layers (32 → 16) with ReLU.
  - Outputs: `z_mean` and `z_log_var` (small latent dimension, e.g. 4–8).
- Sampling:
  - Implementation of the reparameterization trick.
- Decoder:
  - Two dense layers (16 → 32) and linear output of the input size.
 
Using as error the mean MSE per feature as a reconstruction component and the KL divergence with low weight \(\beta\) (0.01–0.1) to avoid penalizing excessively in the early phases.

---

### 3. Retraining pipeline improvement

We propose the following more complete pipeline:

1. **Preprocessing**
   - Scaling with `StandardScaler` on training.
   - k‑NN imputation.
   - PCA if dimensionality is high.

2. **Split**  
   - Train: non-fraud transactions to train AE/VAE.
   - Val/Test: mix of fraud and non-fraud for evaluation.

3. **Baseline**  
   - Classic autoencoder on original pipeline (simple imputation + scaling).
   - Trained and evaluated to reproduce current behavior.

4. **Improved model**  
   - VAE on new pipeline (k‑NN + scaling).
   - Adam optimizer with lower `learning_rate` (`1e-4`) and gradient clipping for stability.
   - Training with early stopping and scheduler (ReduceLROnPlateau).

This pipeline allows for a structured comparison of the base autoencoder with the improved VAE under reproducible conditions.

---

### 4. Evaluation protocol

#### 4.1. Offline evaluation

For each model (baseline and VAE):

- Calculate reconstruction errors in validation and test.
- Set an anomaly threshold, for example the 95th percentile of the error on non-fraud validation transactions.
- In test, convert the score into a binary prediction (fraud/non-fraud) and measure:
  - Fraud recall, precision, and F1.
  - ROC‑AUC and PR‑AUC.

Compare:

- If the improved model increases fraud recall without precision dropping below the acceptable threshold.
- If F1 and PR‑AUC improve compared to the baseline.

#### 4.2. Subgroup evaluation

- Repeat part of the analysis by country/channel/merchant type to check:
  - If false positives are reduced in segments where they were problematic.
  - Or if new biases are introduced (which would need to be mitigated).

---

### 5. Deployment and monitoring plan

The responsible deployment plan is summarized as:

1. **Staging validation**  
   - Execute the VAE with the improved pipeline on recent historical data and compare with the current model.

2. **Gradual deployment**  
   - Introduce the new model in parallel (shadow mode) or on a fraction of the traffic.
   - Maintain rapid rollback capability.

3. **Continuous monitoring**  
   - Monitor technical metrics (recall, precision, F1, PR‑AUC) and business metrics (losses, alert volume, review times).
   - Monitor data drift and fairness; retrain or recalibrate if degradation is detected.

4. **Human supervision**  
   - Include periodic review by analysts and compliance officers to adjust thresholds and the pipeline according to the evolution of fraud.

---

## Closing

Part 1 is as follows:

- **Diagnosis**: the current autoencoder, with simple imputation and without adaptation to drift, no longer fits the current fraud and data environment.
- **Improvement plan**: introduce k-NN imputation and a simple tabular VAE, with careful training and evaluation, aiming to improve fraud recall and F1 and reduce false positives in critical segments.

This approach is what will be implemented, in a simplified manner, in the Part 2 notebook (option A).


</div>